# 05 — SQL Analytics Layer

This notebook adds a **SQL analytics layer** on top of the existing ETL pipeline.
All queries run against `data/superstore.db` (SQLite) and results are visualised using matplotlib.

**Concepts demonstrated:**
- Data ingestion (CSV → SQLite)
- CTEs, window functions, aggregations
- Supply chain KPIs: demand trend, inventory turnover, lead time, stockout risk
- Data quality validation

> Mirrors the kind of SQL work done in enterprise planning platforms like o9 Solutions.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

# ── Setup ──────────────────────────────────────────────────────────────────
DB_PATH      = os.path.join('..', 'data', 'superstore.db')
QUERIES_DIR  = os.path.join('..', 'queries')

plt.rcParams['figure.dpi']     = 120
plt.rcParams['font.family']    = 'sans-serif'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

conn = sqlite3.connect(DB_PATH)
print(f'Connected to {DB_PATH}')

In [ ]:
def run_query(filename):
    """Read a .sql file and return results as a DataFrame."""
    path = os.path.join(QUERIES_DIR, filename)
    with open(path, 'r') as f:
        sql = f.read()
    return pd.read_sql_query(sql, conn)

print('Helper function ready.')

---
## 1. Monthly Demand Trend with Rolling Average
`01_demand_trend.sql` — Window function: `AVG() OVER (ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)`

In [ ]:
df_demand = run_query('01_demand_trend.sql')
print(df_demand.shape)
df_demand.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
categories = df_demand['category'].unique()
colors = {'Technology': '#4C9BE8', 'Furniture': '#E8834C', 'Office Supplies': '#4CB87E'}

for ax, cat in zip(axes, categories):
    sub = df_demand[df_demand['category'] == cat]
    c = colors.get(cat, '#888')
    ax.bar(sub['month'], sub['monthly_units'], color=c, alpha=0.35, label='Monthly units')
    ax.plot(sub['month'], sub['rolling_3m_avg'], color=c, linewidth=2, label='3M rolling avg')
    ax.set_title(cat, fontsize=11, fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Units')
    ax.tick_params(axis='x', rotation=70, labelsize=6)
    ax.legend(fontsize=8)

plt.suptitle('Monthly Demand Trend by Category (with 3-Month Rolling Avg)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/sql_01_demand_trend.png', bbox_inches='tight')
plt.show()

---
## 2. Inventory Turnover by Sub-Category
`02_inventory_turnover.sql` — CTE-based ratio calculation, inventory health classification

In [ ]:
df_turnover = run_query('02_inventory_turnover.sql')
df_turnover.head(10)

In [ ]:
health_colors = {'Fast moving': '#4CB87E', 'Medium moving': '#E8C24C', 'Slow moving': '#E8554C'}
bar_colors = [health_colors.get(h, '#aaa') for h in df_turnover['inventory_health']]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(df_turnover['sub_category'], df_turnover['turnover_ratio'], color=bar_colors)
ax.set_xlabel('Turnover Ratio')
ax.set_title('Inventory Turnover Rate by Sub-Category', fontsize=13, fontweight='bold')
ax.axvline(x=8, color='#4CB87E', linestyle='--', linewidth=1, label='Fast moving threshold (8)')
ax.axvline(x=4, color='#E8C24C', linestyle='--', linewidth=1, label='Medium threshold (4)')
ax.legend(fontsize=8)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in health_colors.items()]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/sql_02_inventory_turnover.png', bbox_inches='tight')
plt.show()

---
## 3. Shipping Lead Time Analysis
`03_lead_time_analysis.sql` — Lead time in days, on-time % by ship mode and region

In [ ]:
df_lead = run_query('03_lead_time_analysis.sql')
df_lead.head(10)

In [ ]:
pivot = df_lead.pivot_table(index='region', columns='ship_mode', values='avg_lead_days', aggfunc='mean')

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='none')
ax.set_title('Avg Lead Time (Days) by Region and Ship Mode', fontsize=13, fontweight='bold')
ax.set_xlabel('Region')
ax.set_ylabel('Avg Lead Days')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Ship Mode', fontsize=8)
ax.axhline(y=3, color='red', linestyle='--', linewidth=1, label='Target: 3 days')

plt.tight_layout()
plt.savefig('../outputs/sql_03_lead_time.png', bbox_inches='tight')
plt.show()

---
## 4. Stockout Risk Detection
`04_stockout_risk.sql` — Rule-based risk flagging using demand vs average comparison

In [ ]:
df_stock = run_query('04_stockout_risk.sql')
df_stock.head(10)

In [ ]:
risk_colors = {
    'HIGH - replenish now':    '#E8554C',
    'MEDIUM - monitor closely': '#E8C24C',
    'LOW - healthy stock signal': '#4CB87E'
}
bar_colors = [risk_colors.get(r, '#aaa') for r in df_stock['stockout_risk']]

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(df_stock['sub_category'], df_stock['demand_vs_avg_pct'], color=bar_colors)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Demand vs Avg (%)')
ax.set_title('Stockout Risk by Sub-Category (Latest Month vs Historical Avg)', fontsize=13, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in risk_colors.items()]
ax.legend(handles=legend_elements, fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/sql_04_stockout_risk.png', bbox_inches='tight')
plt.show()

---
## 5. Profitability vs Discount Impact
`05_profitability_analysis.sql` — RANK() window function, margin analysis by category

In [ ]:
df_profit = run_query('05_profitability_analysis.sql')
df_profit.head(10)

In [ ]:
# Show impact of discounting on margin per category
summary = df_profit.groupby(['category', 'price_type'])['profit_margin_pct'].mean().unstack()

fig, ax = plt.subplots(figsize=(8, 5))
summary.plot(kind='bar', ax=ax, color=['#E8554C', '#4C9BE8'], edgecolor='none')
ax.set_title('Avg Profit Margin %: Discounted vs Full Price by Category', fontsize=12, fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Profit Margin %')
ax.tick_params(axis='x', rotation=0)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.legend(title='Price Type', fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/sql_05_profitability.png', bbox_inches='tight')
plt.show()

---
## 6. Data Quality Validation
`06_data_quality_checks.sql` — Null check, duplicate check, anomaly detection, date logic

In [ ]:
df_dq = run_query('06_data_quality_checks.sql')
print('Data Quality Report:')
df_dq

---
## Summary

| Query | Concept | SQL Features Used |
|---|---|---|
| 01 Demand trend | Demand Management | Window fn, GROUP BY |
| 02 Inventory turnover | Distribution Planning | CTE, NULLIF, CASE |
| 03 Lead time | Supplier planning | julianday(), aggregation |
| 04 Stockout risk | Risk flagging | CTE, JOIN, CASE |
| 05 Profitability | S&OP margin analysis | RANK() OVER PARTITION |
| 06 Data quality | Integration & ingestion | UNION ALL, null checks |

In [ ]:
conn.close()
print('Done. All outputs saved to outputs/ folder.')